# Denoising autoencoder con masking puro, NSL-KDD

Capa propia que anula una fracción de las columnas de entrada sin reescalar el resto, activa solo en entrenamiento. El nivel 0 es el control sin corrupción. Se entrena con KDDTrain+_20Percent y se evalúa con KDDTest+, que incluye tipos de ataque no vistos en entrenamiento.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

In [2]:
COL_NAMES = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]
BASE = Path.home() / ".cache/kagglehub/datasets/hassan06/nslkdd/versions/1"
df_train = pd.read_csv(BASE / "KDDTrain+_20Percent.txt", header=None, names=COL_NAMES).drop(columns=['difficulty'])
df_test = pd.read_csv(BASE / "KDDTest+.txt", header=None, names=COL_NAMES).drop(columns=['difficulty'])

df_train['is_attack'] = (df_train['label'] != 'normal').astype(int)
df_test['is_attack'] = (df_test['label'] != 'normal').astype(int)
df_train['_split'] = 'train'
df_test['_split'] = 'test'
df_all = pd.concat([df_train, df_test], ignore_index=True)

X_all = pd.get_dummies(df_all.drop(columns=['label', 'is_attack', '_split']), columns=['protocol_type', 'service', 'flag'])
is_train = (df_all['_split'] == 'train').to_numpy()

X_train_df = X_all[is_train].reset_index(drop=True)
X_test_df = X_all[~is_train].reset_index(drop=True)
y_train = df_train['is_attack'].to_numpy()
y_test = df_test['is_attack'].to_numpy()
print(f"X_train: {X_train_df.shape}, X_test: {X_test_df.shape}")

X_train: (25192, 119), X_test: (22544, 119)


In [3]:
scaler = MinMaxScaler().fit(X_train_df.to_numpy())
X_train = scaler.transform(X_train_df.to_numpy())
X_test = scaler.transform(X_test_df.to_numpy())
X_train_normal = X_train[y_train == 0]

print(f"X_train: {X_train.shape}, X_train_normal: {X_train_normal.shape}, X_test: {X_test.shape}")

X_train: (25192, 119), X_train_normal: (13449, 119), X_test: (22544, 119)


In [4]:
class PureMasking(layers.Layer):
    """Anula una fracción mask_rate de las columnas sin reescalar el resto, a diferencia de Dropout."""
    def __init__(self, mask_rate, **kwargs):
        super().__init__(**kwargs)
        self.mask_rate = mask_rate

    def call(self, inputs, training=None):
        if training:
            keep_mask = tf.cast(
                tf.random.uniform(tf.shape(inputs)) >= self.mask_rate, inputs.dtype
            )
            return inputs * keep_mask
        return inputs

    def get_config(self):
        config = super().get_config()
        config.update({'mask_rate': self.mask_rate})
        return config


def build_dae_pure_masking(input_dim, hidden_units, mask_rate):
    inputs = keras.Input(shape=(input_dim,))
    x = PureMasking(mask_rate)(inputs)
    for units in hidden_units[:-1]:
        x = layers.Dense(units, activation='relu')(x)
    x = layers.Dense(hidden_units[-1], activation='relu')(x)
    for units in reversed(hidden_units[:-1]):
        x = layers.Dense(units, activation='relu')(x)
    outputs = layers.Dense(input_dim, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='mse')
    return model


def reconstruction_error(model, X):
    return np.mean(np.square(X - model.predict(X, verbose=0)), axis=1)


HIDDEN_UNITS = [64, 32, 16]
MASK_RATES = [0.0, 0.1, 0.2, 0.3]

results = {}
for rate in MASK_RATES:
    tf.keras.utils.set_random_seed(SEED)
    model = build_dae_pure_masking(X_train.shape[1], HIDDEN_UNITS, rate)
    history = model.fit(
        X_train_normal, X_train_normal,
        epochs=100,
        batch_size=64,
        validation_split=0.1,
        callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)],
        verbose=0,
    )
    auc = roc_auc_score(y_test, reconstruction_error(model, X_test))
    results[rate] = {'model': model, 'auc': auc, 'epochs': len(history.history['loss'])}
    print(f"mask_rate={rate}: epocas={results[rate]['epochs']}, ROC-AUC={auc:.4f}")

mask_rate=0.0: epocas=100, ROC-AUC=0.9406


mask_rate=0.1: epocas=100, ROC-AUC=0.9480


mask_rate=0.2: epocas=100, ROC-AUC=0.9399


mask_rate=0.3: epocas=100, ROC-AUC=0.9443


In [5]:
best_rate = max(results, key=lambda k: results[k]['auc'])
best_model = results[best_rate]['model']
print(f"Mejor variante (masking puro): mask_rate={best_rate} (ROC-AUC={results[best_rate]['auc']:.4f})")
print(f"Control (mask_rate=0.0): ROC-AUC={results[0.0]['auc']:.4f}")
print(f"Diferencia mejor masking-puro vs control: {results[best_rate]['auc'] - results[0.0]['auc']:+.4f}")

threshold = np.percentile(reconstruction_error(best_model, X_train_normal), 95)
y_pred = (reconstruction_error(best_model, X_test) > threshold).astype(int)

print()
print(classification_report(y_test, y_pred, target_names=['normal', 'ataque']))
print(confusion_matrix(y_test, y_pred))
print(f"ROC-AUC (mejor variante masking puro): {results[best_rate]['auc']:.4f}")

Mejor variante (masking puro): mask_rate=0.1 (ROC-AUC=0.9480)
Control (mask_rate=0.0): ROC-AUC=0.9406
Diferencia mejor masking-puro vs control: +0.0074



              precision    recall  f1-score   support

      normal       0.83      0.90      0.86      9711
      ataque       0.92      0.86      0.89     12833

    accuracy                           0.88     22544
   macro avg       0.87      0.88      0.87     22544
weighted avg       0.88      0.88      0.88     22544

[[ 8715   996]
 [ 1810 11023]]
ROC-AUC (mejor variante masking puro): 0.9480
